# Akshar AI — Two-Step Detection Pipeline

Trains the Human-vs-Bot ensemble (Step 1) and bot-family classifier (Step 2), renders the geometric evidence plots, and exports the trained artifacts the `akshar-ai` FastAPI service loads from `ml_models/`.

**Run order:** upload your two CSVs to `/content/`, run the pipeline cell, then the export cell to download `akshar_ml_models.zip`.


In [ ]:
# ============================================================================
# TWO-STEP AI DETECTION PIPELINE
# Architecture per research design:
#   Step 1: Binary ensemble — Human vs Bot
#           Two SEPARATE embedding spaces, never concatenated
#           Each space trains its own classifier
#           Meta-learner combines their probability outputs
#   Step 2: Multiclass — Which bot family
#           Style vectors only (bot families differ in HOW they write)
#
# Dataset: Generated column from multi-model dataset (ChatGPT/Grok/Gemini/DeepSeek)
#          Human rows pulled from AIGTxt
#
# NOTE: 6 rows per AI model = proof-of-concept scale only.
#       Geometric visualization is the primary evidence at this scale.
#       Classification metrics are directional, not definitive.
# ============================================================================

# ---- Cell 1: Install ----
!pip install -q sentence-transformers scikit-learn pandas plotly openpyxl umap-learn joblib

# ---- Cell 2: Imports ----
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report)
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================================
# STEP 1: LOAD DATASETS
# ============================================================================

# ---- Load new multi-model dataset ----
# Update path if your file has a different name
ai_raw = pd.read_csv("/content/AI_Generated.csv")  # ← update filename
print("Multi-model dataset shape:", ai_raw.shape)
print(ai_raw.columns.tolist())

# Extract Generated column only — raw AI output, no humanization artifacts
ai_df = ai_raw[["Model", "Model_Version", "Topic", "Generated"]].copy()
ai_df = ai_df.rename(columns={"Generated": "text", "Model": "model_source"})
ai_df["label"]     = "bot"
ai_df["label_fine"] = ai_df["model_source"]  # fine-grained label for Step 2
print("\nAI rows per model:")
print(ai_df["model_source"].value_counts())

# ---- Load AIGTxt for human rows ----
human_raw = pd.read_csv("/content/human_wikipedia.csv")
human_df = human_raw[["text", "topic"]].rename(columns={"topic": "Domain"})
human_df = human_df.dropna().reset_index(drop=True)

n_ai = len(ai_df)
if len(human_df) > n_ai:
    human_df = human_df.sample(n=n_ai, random_state=42).reset_index(drop=True)

human_df["label"]        = "human"
human_df["label_fine"]   = "Human"
human_df["model_source"] = "Human"

print(f"Human rows loaded: {len(human_df)}")
print(human_df["Domain"].value_counts())

# ---- Combine ----
full_df = pd.concat([ai_df[["text", "label", "label_fine", "model_source"]],
                     human_df[["text", "label", "label_fine", "model_source"]]],
                    ignore_index=True)
full_df = full_df.dropna(subset=["text"]).reset_index(drop=True)

print(f"\nFull dataset: {len(full_df)} rows")
print(full_df["label_fine"].value_counts())

# ============================================================================
# STEP 2: LOAD BOTH EMBEDDING MODELS
# ============================================================================

print("\nLoading StyleDistance (768-dim, style space)...")
style_model = SentenceTransformer("StyleDistance/styledistance")

print("Loading BGE (384-dim, semantic space)...")
sem_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# ============================================================================
# STEP 3: EMBED — both models receive same raw text independently
# ============================================================================

def embed_style(texts, batch_size=32):
    return style_model.encode(
        list(texts), batch_size=batch_size,
        normalize_embeddings=True, show_progress_bar=True
    )

def chunk_embed_semantic(texts, batch_size=32):
    def split_sentences(text):
        chunks = re.split(r'(?<=[.!?])\s+', str(text).strip())
        return [c.strip() for c in chunks if c.strip()] or [str(text)]
    doc_embs = []
    for text in texts:
        chunks = split_sentences(text)
        embs = sem_model.encode(chunks, batch_size=batch_size,
                                 normalize_embeddings=True, show_progress_bar=False)
        doc_emb = embs.mean(axis=0)
        norm = np.linalg.norm(doc_emb)
        doc_embs.append(doc_emb / norm if norm > 0 else doc_emb)
    return np.vstack(doc_embs)

print("\nEmbedding all texts through StyleDistance...")
X_style = embed_style(full_df["text"])

print("\nEmbedding all texts through BGE (semantic chunking)...")
X_sem = chunk_embed_semantic(full_df["text"].tolist())

print(f"\nStyle space dim: {X_style.shape[1]}")
print(f"Semantic space dim: {X_sem.shape[1]}")
print("Vectors are SEPARATE — not concatenated at any point")

# Binary labels
y_binary = (full_df["label"] == "bot").astype(int).values  # 1=bot, 0=human

# Fine-grained labels for Step 2
le = LabelEncoder()
y_fine = le.fit_transform(full_df["label_fine"].values)
print(f"\nFine-grained classes: {list(le.classes_)}")

# ============================================================================
# STEP 4: STEP 1 — BINARY ENSEMBLE (Human vs Bot)
# ============================================================================

print("\n" + "="*60)
print("STEP 1: BINARY CLASSIFICATION — Human vs Bot")
print("Using Leave-One-Out CV (honest eval for small datasets)")
print("="*60)

# Style-space classifier
clf_style_bin = RandomForestClassifier(
    n_estimators=200, max_depth=8, random_state=42, n_jobs=-1
)
# Semantic-space classifier
clf_sem_bin = RandomForestClassifier(
    n_estimators=200, max_depth=8, random_state=42, n_jobs=-1
)

# LOO cross-validation to get probability predictions for every row
loo = LeaveOneOut()

p_style_loo = cross_val_predict(clf_style_bin, X_style, y_binary,
                                  cv=loo, method="predict_proba")[:, 1]
p_sem_loo   = cross_val_predict(clf_sem_bin,   X_sem,   y_binary,
                                  cv=loo, method="predict_proba")[:, 1]

# Meta-learner: takes [P_style, P_sem] → final binary prediction
X_meta = np.column_stack([p_style_loo, p_sem_loo])
meta_clf = LogisticRegression(random_state=42)
p_meta_loo = cross_val_predict(meta_clf, X_meta, y_binary,
                                 cv=loo, method="predict_proba")[:, 1]
preds_binary = (p_meta_loo >= 0.5).astype(int)

acc_binary = accuracy_score(y_binary, preds_binary)
cm_binary  = confusion_matrix(y_binary, preds_binary)
fp_rate    = cm_binary[0,1] / (cm_binary[0,0] + cm_binary[0,1])

print(f"\nBinary Ensemble (LOO-CV):")
print(f"  Accuracy:           {acc_binary:.3f}")
print(f"  False-positive rate: {fp_rate:.3f}  (humans wrongly flagged)")
print(f"  Confusion matrix [[TN,FP],[FN,TP]]:\n  {cm_binary}")

# Also train full models (on all data) for Step 2 use and visualization
clf_style_bin.fit(X_style, y_binary)
clf_sem_bin.fit(X_sem, y_binary)
p_style_full = clf_style_bin.predict_proba(X_style)[:, 1]
p_sem_full   = clf_sem_bin.predict_proba(X_sem)[:, 1]
X_meta_full  = np.column_stack([p_style_full, p_sem_full])
meta_clf.fit(X_meta_full, y_binary)

# ============================================================================
# STEP 5: STEP 2 — MULTICLASS (Which Bot Family)
# ============================================================================

print("\n" + "="*60)
print("STEP 2: MULTICLASS — Which Bot Family?")
print("(AI rows only, Style space, LOO-CV)")
print("="*60)

ai_mask    = full_df["label"] == "bot"
X_style_ai = X_style[ai_mask]
y_ai_fine  = le.transform(full_df[ai_mask]["label_fine"].values)
ai_classes = full_df[ai_mask]["label_fine"].values

clf_multi = RandomForestClassifier(
    n_estimators=200, max_depth=6, random_state=42, n_jobs=-1
)
preds_multi_loo = cross_val_predict(clf_multi, X_style_ai, y_ai_fine, cv=loo)
acc_multi = accuracy_score(y_ai_fine, preds_multi_loo)

print(f"\nBot Family Classification (LOO-CV):")
print(f"  Accuracy: {acc_multi:.3f}")
print(f"  Classes: {list(le.classes_)}")
print(f"\nDetailed breakdown:")
print(classification_report(
    y_ai_fine, preds_multi_loo,
    target_names=[c for c in le.classes_ if c != "Human"]
))

# Train full model for visualization
clf_multi.fit(X_style_ai, y_ai_fine)

# ============================================================================
# STEP 6: VISUALIZATIONS
# ============================================================================

# ---- Plot 1: Meta-learner decision space ----
viz_df = full_df.copy()
viz_df["P_style_bot"] = p_style_full
viz_df["P_sem_bot"]   = p_sem_full
viz_df["P_meta_bot"]  = meta_clf.predict_proba(X_meta_full)[:, 1]
viz_df["binary_pred"] = (viz_df["P_meta_bot"] >= 0.5).map({True: "Bot", False: "Human"})
viz_df["correct"]     = viz_df["binary_pred"] == viz_df["label"].str.capitalize()
viz_df["snippet"]     = viz_df["text"].str.slice(0, 100) + "..."

fig1 = px.scatter(
    viz_df, x="P_style_bot", y="P_sem_bot",
    color="label_fine", symbol="correct",
    hover_data={"snippet": True, "P_meta_bot": ":.3f",
                "binary_pred": True, "correct": True,
                "P_style_bot": False, "P_sem_bot": False},
    title="Step 1 — Meta-Learner Input Space<br>"
          "<sup>X = StyleDistance P(bot) | Y = BGE P(bot) | "
          "Each point = one text | Two spaces never merged</sup>",
    color_discrete_map={
        "Human": "#2ecc71", "ChatGPT": "#e74c3c",
        "Grok": "#3498db", "Gemini": "#f39c12", "DeepSeek": "#9b59b6"
    },
    width=850, height=600,
)
fig1.add_vline(x=0.5, line_dash="dash", line_color="gray", opacity=0.5)
fig1.add_hline(y=0.5, line_dash="dash", line_color="gray", opacity=0.5)
fig1.update_traces(marker=dict(size=12, line=dict(width=1, color="DarkSlateGrey")))
fig1.update_layout(template="plotly_white")
fig1.show()

# ---- Plot 2: PCA of Style space — all classes ----
pca2 = PCA(n_components=2, random_state=42)
coords2 = pca2.fit_transform(X_style)
viz_df["pca_x"] = coords2[:, 0]
viz_df["pca_y"] = coords2[:, 1]

fig2 = px.scatter(
    viz_df, x="pca_x", y="pca_y",
    color="label_fine",
    hover_data={"snippet": True, "label_fine": True,
                "pca_x": False, "pca_y": False},
    title="StyleDistance Space — PCA (All Classes)<br>"
          "<sup>Do bot families cluster separately from humans and from each other?</sup>",
    color_discrete_map={
        "Human": "#2ecc71", "ChatGPT": "#e74c3c",
        "Grok": "#3498db", "Gemini": "#f39c12", "DeepSeek": "#9b59b6"
    },
    labels={"pca_x": f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})",
            "pca_y": f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})"},
    width=850, height=600,
)
fig2.update_traces(marker=dict(size=14, opacity=0.85,
                                line=dict(width=1.5, color="DarkSlateGrey")))
fig2.update_layout(template="plotly_white")
fig2.show()

# ---- Plot 3: Bot families only — zoomed PCA ----
ai_viz = viz_df[viz_df["label"] == "bot"].copy()
fig3 = px.scatter(
    ai_viz, x="pca_x", y="pca_y", color="label_fine",
    hover_data={"snippet": True, "pca_x": False, "pca_y": False},
    title="Bot Families Only — StyleDistance PCA<br>"
          "<sup>Each model should occupy a distinct region if style fingerprints differ</sup>",
    color_discrete_map={
        "ChatGPT": "#e74c3c", "Grok": "#3498db",
        "Gemini": "#f39c12",  "DeepSeek": "#9b59b6"
    },
    width=850, height=550,
)
fig3.update_traces(marker=dict(size=16, opacity=0.9,
                                line=dict(width=1.5, color="DarkSlateGrey")))
fig3.update_layout(template="plotly_white")
fig3.show()

# ---- Plot 4: Summary accuracy comparison ----
summary_df = pd.DataFrame({
    "Step": ["Step 1: Human vs Bot\n(Binary Ensemble)",
             "Step 2: Bot Family\n(Multiclass Style)"],
    "Accuracy": [acc_binary, acc_multi],
    "N rows": [len(full_df), int(ai_mask.sum())],
})
fig4 = px.bar(
    summary_df, x="Step", y="Accuracy", color="Step",
    text="Accuracy", title="Pipeline Accuracy — Both Steps (LOO-CV)",
    color_discrete_sequence=["#2ecc71", "#e74c3c"],
    height=420,
)
fig4.update_traces(texttemplate="%{text:.3f}", textposition="outside",
                   marker_line_width=0)
fig4.update_layout(template="plotly_white", yaxis_range=[0, 1.1],
                   showlegend=False)
fig4.show()

print("\nPipeline trained. Run the next cell to export artifacts for the akshar-ai service.")

In [ ]:
# ============================================================================
# EXPORT ARTIFACTS FOR THE akshar-ai SERVICE
# Saves the trained classifiers in the exact filenames the FastAPI service
# loads from `akshar-app/packages/ai/ml_models/`. Downloads a single zip.
# ============================================================================
import os, json, joblib
from datetime import datetime, timezone

OUT_DIR = "/content/ml_models"
os.makedirs(OUT_DIR, exist_ok=True)

# Primary artifact: single-file bundle the akshar-ai service loads (detector_v1.pkl)
joblib.dump({
    "style_rf": clf_style_bin,
    "sem_rf":   clf_sem_bin,
    "meta_lr":  meta_clf,
    "style_model": "StyleDistance/styledistance",
    "sem_model":   "BAAI/bge-small-en-v1.5",
    "threshold": 0.5,
}, f"{OUT_DIR}/detector_v1.pkl")

# Also emit individual files (legacy layout + optional Step-2 family model)
joblib.dump(clf_style_bin, f"{OUT_DIR}/rf_style_binary.pkl")     # Step 1: style RF
joblib.dump(clf_sem_bin,   f"{OUT_DIR}/rf_semantic_binary.pkl")  # Step 1: semantic RF
joblib.dump(meta_clf,      f"{OUT_DIR}/meta_learner.pkl")        # Step 1: meta-learner
joblib.dump(clf_multi,     f"{OUT_DIR}/rf_bot_family.pkl")       # Step 2: bot family
joblib.dump(le,            f"{OUT_DIR}/label_encoder.pkl")       # label encoder

manifest = {
    "version": datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "style_model": "StyleDistance/styledistance",
    "semantic_model": "BAAI/bge-small-en-v1.5",
    "style_dim": int(X_style.shape[1]),
    "semantic_dim": int(X_sem.shape[1]),
    "n_rows": int(len(full_df)),
    "n_human": int((full_df["label"] == "human").sum()),
    "n_bot": int((full_df["label"] == "bot").sum()),
    "bot_families": [c for c in le.classes_ if c != "Human"],
    "classes": list(le.classes_),
    "metrics": {"binary_loo_accuracy": round(float(acc_binary), 4),
                "binary_loo_fp_rate": round(float(fp_rate), 4),
                "bot_family_loo_accuracy": round(float(acc_multi), 4)},
    "artifacts": ["detector_v1.pkl", "rf_style_binary.pkl", "rf_semantic_binary.pkl",
                  "meta_learner.pkl", "rf_bot_family.pkl", "label_encoder.pkl"],
}
with open(f"{OUT_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved:")
for a in manifest["artifacts"] + ["manifest.json"]:
    print("  ", a)

# Zip + download
import shutil
from google.colab import files
shutil.make_archive("/content/akshar_ml_models", "zip", OUT_DIR)
print("\nUnzip the download into akshar-app/packages/ai/ml_models/ and commit it.")
files.download("/content/akshar_ml_models.zip")

In [ ]:
# Save all four plots as HTML and download them
from google.colab import files

fig1.write_html("/content/plot1_meta_learner_space.html")
fig2.write_html("/content/plot2_pca_all_classes.html")
fig3.write_html("/content/plot3_bot_families_only.html")
fig4.write_html("/content/plot4_accuracy_summary.html")

files.download("/content/plot1_meta_learner_space.html")
files.download("/content/plot2_pca_all_classes.html")
files.download("/content/plot3_bot_families_only.html")
files.download("/content/plot4_accuracy_summary.html")